# Forecasting California Retail Gasoline Prices, 1–12 Weeks Ahead
**UCLA Anderson MSBA · Time Series Analysis & Forecasting · Group Project (2026)**

**Research question:** *Can we forecast California retail gasoline prices 1–12 weeks ahead, and what drives them?*

This notebook runs the whole project end to end. Each step calls the same functions as `analysis.py`, which live in `src/`:

| Step | What | Module |
|---|---|---|
| 1 | Profile the 8 EIA series | `src/data.py` |
| 2 | Build a weekly panel with **no look-ahead** and test it | `src/data.py` |
| 3 | Exploratory story: the 2026 crude shock, California's premium, seasonality | `src/plots.py` |
| 4 | Models: random walk, seasonal naive and seasonal mean, ARIMA, ETS, Ridge, GBM, ensembles | `src/models.py` |
| 5 | Rolling-origin backtest, Jan 2022 to latest, refit every 4 weeks | `src/backtest.py` |
| 6 | Validation (2022–24) → model choice → untouched holdout (2025–26), regimes, DM tests | `src/evaluate.py` |
| 7 | 12-week forecast with empirical 80% and 90% intervals; Brent scenario | `src/backtest.py`, `src/scenario.py` |
| 8 | Option A: honest LLM-forecast comparison kit | `llm_kit.py` |

The full backtest takes about 4 minutes. Set `USE_CACHE = True` to reuse `outputs/backtest_forecasts.csv`.

In [ ]:
import json, time, warnings
import numpy as np, pandas as pd
from IPython.display import Image, display, Markdown

from src import config as C, data as D, models as M, backtest as B, evaluate as E, plots as P, scenario as S
from analysis import lookahead_mutation_test

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
C.set_seeds()
C.FIG_DIR.mkdir(parents=True, exist_ok=True)
USE_CACHE = False
show = lambda path: display(Image(filename=str(path), width=950))

## 1. Data profile
The folder contains eight EIA series. Retail prices are weekly on Mondays. Crude is daily, and its blanks are market holidays, which we drop. Inventories and refinery utilisation are weekly on Fridays; PADD5 utilisation only starts in June 2010.

In [ ]:
raw = D.load_raw()
prof = D.profile(raw)
prof

## 2. Weekly panel with no look-ahead
Everything is indexed on the **Monday retail-price date T** (the forecast origin).

* **Crude:** the last close on or before the prior Friday (T − 3 days).
* **EIA stocks and utilisation:** the week ending Friday F is released the following Wednesday, so we use it only from Monday **F + 10**.
* **Features:** past CA log changes (1w, 4w); Brent log changes (1w, 4w, 12w); the retail-minus-crude margin (CA − Brent/42) minus its 52-week mean; the CA–US spread minus its 52-week mean; PADD5 and US inventories as % deviation from the same-week 5-year average; PADD5 utilisation and its 4-week change; week-of-year sin/cos.
* **Seasonal expectation `seas_h`:** the average log change over the next *h* weeks at this week-of-year in the prior 10 calendar years. Every term ends at least 40 weeks before T.
* **Targets** `tgt_h = ln(CA_{T+h}) − ln(CA_T)`.

In [ ]:
panel = D.build_panel(raw)
print(panel.shape, panel.index[0].date(), "→", panel.index[-1].date())
panel[D.FEATURES_BASE + ["seas_4", "tgt_4"]].tail(4).T.round(4)

**Look-ahead test.** For 26 origins we rebuild the panel from raw data truncated to what was *published* by that origin, and require the features to be identical to the full-sample panel. We then test the test: if EIA data are aligned with a 3-day lag instead of the true 10 days, the check must fail.

In [ ]:
la = D.check_no_lookahead(panel, raw)
print(f"{la.ok.sum()}/{len(la)} origins identical (max abs diff {la.max_abs_diff.max():.1e})")
print("Leak-injection test caught the leak:", lookahead_mutation_test(raw, panel))

## 3. The story in the data
### 3a. The 2026 crude shock and California's premium
Brent went from about \$62 in December 2025 to more than \$125 in March–April 2026. It eased to roughly \$70 in June and surged again to about \$135 by early October. California pump prices followed, peaking near \$6 in May and reaching \$6.23 in the latest week. To keep a single y-axis, we plot crude in $/gal (Brent ÷ 42); the vertical gap between the lines is the retail margin.

In [ ]:
show(P.fig_prices(panel))

In [ ]:
show(P.fig_premium(panel))

California's premium over the US average rose from about \$0.25 in the mid-2000s to about \$1.25 in 2025. Its spikes coincide with West Coast refinery outages. The causes are structural: California's own cleaner-burning blend, higher taxes and fees, and an isolated refining market.

### 3b. Seasonality: the strongest single signal

In [ ]:
show(P.fig_seasonal(panel))

On average, prices climb from February to May (refinery maintenance, then the switch to summer blend) and fall from October to December (winter blend and lower demand). Seasonality was the strongest signal in our first pass, which is why we build the explicit *seasonal expectation* feature.

### 3c. Margins absorb crude shocks; inventories carry a weak signal

In [ ]:
show(P.fig_margin(panel))

In [ ]:
show(P.fig_inventory(panel))

In [ ]:
show(P.fig_corr(panel))

* When crude jumps, the margin is squeezed first and retail catches up over the following weeks ("rockets and feathers"). The margin deviation is a mean-reverting predictor: its correlation with the next 4-week change is −0.37.
* Brent momentum (4w change) and the seasonal expectation are the most positively correlated predictors of the 4-week-ahead change.
* The inventory signal is weak (correlation of about 0.2) and partly seasonal. Stocks are seasonally high in late winter, just before the spring run-up.

## 4. Models
| Model | Description |
|---|---|
| Random walk | No change. The benchmark. |
| Seasonal naive | Last year's change over the same weeks, applied to today's level |
| Seasonal mean (10y) | The `seas_h` feature itself, i.e. the average same-season change over the prior 10 years |
| ARIMA(p,1,q) | Order chosen by AIC on **pre-2022 data only**; parameters re-estimated at each refit |
| ETS damped | Holt additive damped trend on log price |
| Ridge / GBM | **Direct** multi-horizon regressions, one per h, with and without the seasonal feature |
| Ensembles | Equal-weight averages of log-change forecasts, with members fixed in advance |

In [ ]:
aic = M.select_arima_order(panel)
best = aic.iloc[0]
order, trend = (int(best.p), 1, int(best.q)), best.trend
print("Selected ARIMA order:", order, "trend:", trend)
aic.head(8).round(1)

## 5. Rolling-origin backtest
* Weekly origins from 3 Jan 2022 to the latest date, with an expanding window and parameters re-estimated every 4 weeks.
* Direct regressions train only on rows whose **target date ≤ origin**. This is asserted inside `DirectRegression.fit`.
* All 12 horizons are forecast, because the fan chart needs them. Tables report h = 1, 4, 8, 12.

In [ ]:
models = M.make_models(order, trend)
cache = C.OUT_DIR / "backtest_forecasts.csv"
t0 = time.time()
if USE_CACHE and cache.exists():
    bt = pd.read_csv(cache, parse_dates=["origin", "target_date"])
else:
    bt = B.run_backtest(panel, models)
    bt.to_csv(cache, index=False)
print(f"{bt.origin.nunique()} origins × {bt.model.nunique()} models × 12 horizons in {time.time()-t0:.0f}s")

## 6. Validation → model choice → holdout
We choose the top model by mean RMSE relative to the random walk over h ∈ {1, 4, 8, 12}, **on targets in 2022–2024 only**. We then report its accuracy on 2025–2026 targets, which played no part in the choice.

In [ ]:
t_valid = E.accuracy_table(bt, E.by_year(*C.VALID_YEARS))
t_hold = E.accuracy_table(bt, E.by_year(*C.HOLDOUT_YEARS))
top, ranking = E.select_top_model(t_valid)
ranking = ranking.join(t_hold.groupby("model").RelRMSE.mean().rename("holdout_mean_RelRMSE"))
print("Top model (validation):", top)
ranking.round(3)

In [ ]:
display(Markdown("**Validation 2022–24: RMSE relative to random walk**")); display(E.wide(t_valid).round(3))
display(Markdown("**Holdout 2025–26: RMSE relative to random walk**")); display(E.wide(t_hold).round(3))

In [ ]:
cols = ["model", "h", "n", "MAE", "RMSE", "MAPE", "DirAcc", "RelRMSE", "DM_stat", "DM_p"]
t_hold[t_hold.model.isin([top, "Random walk", "Seasonal mean (10y)", "ARIMA", "Ridge + seasonal"])][cols].round(3)

MAE and RMSE are in $/gal. DirAcc is the share of forecasts that called the direction of change correctly; it is undefined for the random walk. DM is the Diebold–Mariano test against the random walk, using squared-error loss, Newey–West variance with lag h−1 and the Harvey–Leybourne–Newbold correction. A negative statistic means the model beats the random walk.

In [ ]:
show(P.fig_error_by_horizon(E.accuracy_table(bt, E.by_year(*C.VALID_YEARS), C.HORIZONS),
                            E.accuracy_table(bt, E.by_year(*C.HOLDOUT_YEARS), C.HORIZONS), top))

In [ ]:
show(P.fig_backtest(bt, panel, top))

### Regimes: the 2022 spike, 2023–25 normal, and the 2026 shock

In [ ]:
reg = E.regime_table(bt)
display(reg[reg.model.isin([top, "Seasonal mean (10y)", "ARIMA", "Ridge + seasonal", "Random walk"])]
        .pivot_table(index="model", columns=["regime", "h"], values="RelRMSE").round(2))
show(P.fig_regimes(reg, top))

## 7. Final 12-week forecast
Every model is refit on all data to the latest week. The top model's 80% and 90% bands are the empirical 10/90 and 5/95 quantiles of its backtest log errors at each horizon. As an honest check, bands built **only from 2022–24 errors** are tested for coverage on the 2025–26 holdout.

In [ ]:
final, fitted = B.final_forecast(panel, models)
q = B.interval_quantiles(bt, top)
fan = B.add_intervals(final, q, top)
display(fan[["h", "target_date", "fc_price", "lo80", "hi80", "lo90", "hi90"]].round(3))
q_val = B.interval_quantiles(bt[bt.target_date.dt.year <= 2024], top)
h_rows = bt[(bt.model == top) & bt.actual_price.notna() & (bt.target_date.dt.year >= 2025)]
err = np.log(h_rows.actual_price) - np.log(h_rows.fc_price)
cov = pd.DataFrame({"cover80": (err >= h_rows.h.map(q_val.q10)) & (err <= h_rows.h.map(q_val.q90)),
                    "cover90": (err >= h_rows.h.map(q_val.q05)) & (err <= h_rows.h.map(q_val.q95)),
                    "h": h_rows.h}).groupby("h").mean().mul(100).round(1)
print("Holdout coverage (%) of bands built from 2022-24 errors:"); display(cov.T)
show(P.fig_fan(panel, final, fan, top))

In [ ]:
final.pivot(index='h', columns='model', values='fc_price').loc[[1, 4, 8, 12]].T.round(2)

## 8. What drives the forecast?

In [ ]:
ridge, gbm = fitted["Ridge + seasonal"], fitted["GBM + seasonal"]
show(P.fig_importance(ridge.coefficients(), gbm.coefficients()))

**Brent scenario.** We re-run the Ridge driver model as if Brent had moved ±X% in the latest week. All three Brent-change features rise by ln(1+X), and the margin deviation falls by Brent·X/42 because pump prices have not adjusted yet. The *full pass-through* column is the mechanical crude-cost change per gallon. The same calculation powers the slider in the Streamlit app.

In [ ]:
scen = pd.concat([S.brent_scenario(panel, ridge, x).assign(brent_move_pct=x) for x in (-20, -10, 10, 20)])
scen[scen.h.isin([1, 4, 8, 12])].pivot_table(index="brent_move_pct", columns="h", values="difference").round(3)

## 9. AI component
**Option A: LLM-forecast comparison kit (`llm_kit.py`).** For 20 origins, 13 before and 7 after the model's stated knowledge cutoff, it writes a prompt containing **only** the data published by that origin. This is built with the same truncation function that passed the look-ahead test. The prompt asks for 1/4/8/12-week forecasts in JSON. If `ANTHROPIC_API_KEY` is set, the kit calls Claude with structured JSON output. Otherwise it saves the prompts, so the team can paste them into fresh chats with no memory and no web search. The answers are scored with the same metrics and DM tests as the models, separately for pre-cutoff origins (contamination risk) and post-cutoff origins (clean).

**Option B: Streamlit app (`app.py`).** It provides history charts with a driver selector, model and horizon pickers, backtest forecasts against actuals, the accuracy table, the 12-week forecast with bands, and a Brent ±X% scenario slider.

In [ ]:
import llm_kit
manifest = llm_kit.choose_origins(panel)
print(manifest.post_cutoff.value_counts().rename({True: "post-cutoff", False: "pre-cutoff"}))
print(llm_kit.build_prompt(raw, manifest.origin.iloc[-1])[:1400], "...")
if any((C.LLM_DIR / "responses").glob("*.*")):
    display(llm_kit.score())
else:
    print("No LLM responses saved yet: run `python llm_kit.py prepare`, then `python llm_kit.py score`.")

## 10. Headline results

In [ ]:
def row(t, m, h): return t[(t.model == m) & (t.h == h)].iloc[0]
lines = [f"**Top model:** {top} (chosen on 2022–24 validation)", "", "| h | holdout MAE | RW MAE | RelRMSE | DM p |", "|---|---|---|---|---|"]
for h in C.REPORT_H:
    a, r = row(t_hold, top, h), row(t_hold, "Random walk", h)
    lines.append(f"| {h} | ${a.MAE:.3f} | ${r.MAE:.3f} | {a.RelRMSE:.3f} | {a.DM_p:.3f} |")
f12 = fan.iloc[-1]
lines += ["", f"**12-week forecast ({f12.target_date:%b %d, %Y}):** ${f12.fc_price:.2f}/gal "
          f"(80%: ${f12.lo80:.2f}–${f12.hi80:.2f}; 90%: ${f12.lo90:.2f}–${f12.hi90:.2f}), from ${panel.ca.iloc[-1]:.2f} on {panel.index[-1]:%b %d}."]
display(Markdown("\n".join(lines)))

**Takeaways**
1. **Seasonality is the backbone.** A 10-year seasonal mean alone cuts 12-week RMSE by 22–24% relative to the random walk, in both validation and holdout.
2. **Drivers add value at short horizons.** Crude momentum, margin and spread deviations, and recent CA momentum, fed through Ridge and GBM, give the biggest gains at 1–4 weeks. The ML ensemble with the seasonal feature was best on validation and beat the random walk at every horizon on the holdout. Only the 1-week gain is statistically significant there (DM p ≈ 0.005); with about 90 overlapping holdout errors, the 4–12-week gains are not.
3. **Shocks change the ranking.** In the 2026 crude shock, errors were about twice those of calm years. The plain seasonal mean beat the ML ensemble at 4–12 weeks, because crude momentum reversed sharply in June and again in September.
4. **The intervals are honest.** Bands built from 2022–24 errors covered close to their nominal share of 2025–26 outcomes, slightly over-covering at middle horizons.